# Загрузка данных

In [2]:
import nltk
import re
from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups

In [ ]:
nltk.download('punkt_tab') # токенизация
nltk.download('wordnet') # лематизация
nltk.download('averaged_perceptron_tagger') # обученная модель для POS tagger
nltk.download('stopwords') # загружаем список стоп слов

In [33]:
from nltk.corpus import wordnet
from nltk.corpus import stopwords
from nltk import pos_tag

In [5]:
dataset_emotion_train = load_dataset('emotion', split='train')

In [6]:
categories = ['comp.sys.ibm.pc.hardware',
'comp.sys.mac.hardware',
'comp.graphics',
'comp.windows.x',
]
dataset_news_train = fetch_20newsgroups(subset='train',categories=categories)

In [7]:
stop_words = set(stopwords.words('english'))

# Удаление спец. символов и слов разметки

In [8]:
# количество первых примеров для анализа
n_1 = 20
n_2 = 5

In [9]:
for i in range(n_1):
    print(dataset_emotion_train['text'][i])

i didnt feel humiliated
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
im grabbing a minute to post i feel greedy wrong
i am ever feeling nostalgic about the fireplace i will know that it is still on the property
i am feeling grouchy
ive been feeling a little burdened lately wasnt sure why that was
ive been taking or milligrams or times recommended amount and ive fallen asleep a lot faster but i also feel like so funny
i feel as confused about life as a teenager or as jaded as a year old man
i have been with petronas for years i feel that petronas has performed well and made a huge profit
i feel romantic too
i feel like i have to make the suffering i m seeing mean something
i do feel that running is a divine experience and that i can expect to have some type of spiritual encounter
i think it s the easiest time of year to feel dissatisfied
i feel low energy i m just thirsty
i have immense sympathy with the general point but a

In [10]:
for i in range(n_2):
    print(dataset_news_train.data[i])

From: bgrubb@dante.nmsu.edu (GRUBB)
Subject: Re: IDE vs SCSI
Organization: New Mexico State University, Las Cruces, NM
Lines: 49
Distribution: world
NNTP-Posting-Host: dante.nmsu.edu

DXB132@psuvm.psu.edu writes:
>SCSI-I ranges from 0-5MB/s.
>SCSI-II ranges from 0-40MB/s.
>IDE ranges from 0-8.3MB/s.                                       
>ESDI is always 1.25MB/s (although there are some non-standard versions)
The above does not tell the proper story of SCSI:
SCSI-I: 8-bit asynchronous {~1.5MB/s ave}, synchronous {5MB/s max} transfer 
base.
SCSI-1{faster} this requires a SCSI-2 controller chip and provides
 SCSI-2 {8-bit to 16-bit} speeds with SCSI-1 controlers.
SCSI-2: 4-6MB/s with 10MB/s burst{8-bit}, 8-12MB/s with 20MB/s burst {16-bit}, 
and 15-20MB/s with 40MB/s burst{32-bit/wide and fast}.  16-bit SCSI can be
wide or fast, it depends on how the port is designed{The Quadras will support
fast SCSI but not wide when the OS SCSI manager is rewritten since the
Quardas use a SCSI-1 {non-

В первом датасете нет спец символов и спец слов.

Во втором датасете видим следующие спец. символы:
1. Заголовки писем (метаданные)
    ```text
    From: bgrubb@dante.nmsu.edu (GRUBB)
    Subject: Re: IDE vs SCSI
    Organization: New Mexico State University, Las Cruces, NM
    Lines: 49
    Distribution: world
    NNTP-Posting-Host: dante.nmsu.edu
    ```
2. Электронные адреса и имена
3. Цитаты предыдущих сообщений
    ```text
   >SCSI-I ranges from 0-5MB/s.
   >SCSI-II ranges from 0-40MB/s.
    ```
4.  Разделители и графика
    ```text
    ======================================================================
    ------
    _________________________________________________________________
     /*---------*\*/*-------------------------------------------*\
     *|  ____/|  *|*    PETER.VANDERVEEN@VISSER.EL.WAU.NL       |*
     *|  \ o.O|  *|*    Department of Genetics                  |*
     *|   =(_)=  *|*    Agricultural University                 |*
     *|     U    *|*    Wageningen, The Netherlands             |*
     \*---------*/*\*-------------------------------------------*/
    ```

In [28]:
def clean_text(text):
    # 1. Удаляем заголовки писем (From:, Subject: и т.д.)
    text = re.sub(r'^(From|Subject|Organization|Lines|Distribution|NNTP-Posting-Host|Keywords|Summary):.*$', '', text, flags=re.MULTILINE)

    # 2. Удаляем строки, начинающиеся с '>' (цитаты)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)

    # 3. Удаляем строки, состоящие в основном из повторяющихся символов (===, ---, *** и т.п.)
    #    (например, строки, где более 80% символов — это знаки препинания или пробелы)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        # Если в строке больше 70% не-буквенно-цифровых символов, пропускаем её
        if len(line) == 0:
            continue
        non_alnum = sum(1 for c in line if not c.isalnum() and not c.isspace())
        if non_alnum / len(line) > 0.7 and len(line) > 10:
            continue
        cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)

    # 4. Удаляем электронные адреса
    text = re.sub(r'\S+@\S+', '', text)

    # 5. Удаляем оставшиеся специальные символы, но оставляем буквы, цифры и базовую пунктуацию
    text = re.sub(r"[^\w\s.!?']", ' ', text)

    # 6. Приводим множественные пробелы к одному и убираем пробелы в начале/конце
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [29]:
dataset_news_train_cleaned = [clean_text(text) for text in dataset_news_train.data]

In [30]:
for i in range(n_2):
    print(dataset_news_train_cleaned[i])

writes The above does not tell the proper story of SCSI SCSI I 8 bit asynchronous 1.5MB s ave synchronous 5MB s max transfer base. SCSI 1 faster this requires a SCSI 2 controller chip and provides SCSI 2 8 bit to 16 bit speeds with SCSI 1 controlers. SCSI 2 4 6MB s with 10MB s burst 8 bit 8 12MB s with 20MB s burst 16 bit and 15 20MB s with 40MB s burst 32 bit wide and fast . 16 bit SCSI can be wide or fast it depends on how the port is designed The Quadras will support fast SCSI but not wide when the OS SCSI manager is rewritten since the Quardas use a SCSI 1 non wide port . The article in PC Mag 4 27 93 29 was talking about SCSI 1 SCSI 2 uses TEN 10 devices in it native mode outside its native mode it behaves a lot like SCSI 1 7 devices slower through put From your own figures SCSI 1 is indeed twice ESDI as the article pointed out as for 20 faster then IDE that seems to be 8 bit SCSI 1 using a SCSI 2 contoler chip The Mac Quadra uses a SCSI 2 controler chip for its SCSI 1 and gets 6M

# Удаление слов по частям речи

In [31]:
for i in range(n_1):
    print(dataset_emotion_train['text'][i])

i didnt feel humiliated
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
im grabbing a minute to post i feel greedy wrong
i am ever feeling nostalgic about the fireplace i will know that it is still on the property
i am feeling grouchy
ive been feeling a little burdened lately wasnt sure why that was
ive been taking or milligrams or times recommended amount and ive fallen asleep a lot faster but i also feel like so funny
i feel as confused about life as a teenager or as jaded as a year old man
i have been with petronas for years i feel that petronas has performed well and made a huge profit
i feel romantic too
i feel like i have to make the suffering i m seeing mean something
i do feel that running is a divine experience and that i can expect to have some type of spiritual encounter
i think it s the easiest time of year to feel dissatisfied
i feel low energy i m just thirsty
i have immense sympathy with the general point but a

## Убираем всё кроме сущ. и прил.

In [35]:
tokens_emotion = []
for i in range(len(dataset_emotion_train)):
    tokens_emotion.append(nltk.word_tokenize(dataset_emotion_train['text'][i]))

In [41]:
nouns_adjs_emotion = []
tagged_tokens_emotion = []
for i in range(len(dataset_emotion_train)):
    tagged_tokens_emotion.append(pos_tag(tokens_emotion[i]))
    nouns_adjs = [word for word, tag in tagged_tokens_emotion[i] if tag.startswith('N') or tag.startswith('J')]
    nouns_adjs_emotion.append(nouns_adjs)

In [49]:
print(nouns_adjs_emotion[:10])

[['i', 'feel'], ['i', 'hopeless', 'damned', 'hopeful', 'someone', 'awake'], ['im', 'minute', 'i', 'feel', 'greedy', 'wrong'], ['i', 'nostalgic', 'fireplace', 'i', 'property'], ['i', 'grouchy'], ['ive', 'little', 'burdened', 'wasnt', 'sure'], ['ive', 'milligrams', 'times', 'recommended', 'amount', 'ive', 'lot', 'funny'], ['i', 'life', 'teenager', 'year', 'old', 'man'], ['i', 'petronas', 'years', 'i', 'petronas', 'huge', 'profit'], ['i', 'romantic']]


## Убираем всё кроме сущ., прил. и глаголов

In [43]:
nouns_adjs_verbs_emotion = []
for i in range(len(dataset_emotion_train)):
    nouns_adjs_verbs = [word for word, tag in tagged_tokens_emotion[i] if tag.startswith('N') or tag.startswith('J') or tag.startswith('V')]
    nouns_adjs_verbs_emotion.append(nouns_adjs_verbs)

In [50]:
print(nouns_adjs_verbs_emotion[:10])

[['i', 'didnt', 'feel', 'humiliated'], ['i', 'go', 'feeling', 'hopeless', 'damned', 'hopeful', 'being', 'someone', 'cares', 'is', 'awake'], ['im', 'grabbing', 'minute', 'post', 'i', 'feel', 'greedy', 'wrong'], ['i', 'am', 'feeling', 'nostalgic', 'fireplace', 'i', 'know', 'is', 'property'], ['i', 'am', 'feeling', 'grouchy'], ['ive', 'been', 'feeling', 'little', 'burdened', 'wasnt', 'sure', 'was'], ['ive', 'been', 'taking', 'milligrams', 'times', 'recommended', 'amount', 'ive', 'fallen', 'asleep', 'lot', 'i', 'feel', 'funny'], ['i', 'feel', 'confused', 'life', 'teenager', 'jaded', 'year', 'old', 'man'], ['i', 'have', 'been', 'petronas', 'years', 'i', 'feel', 'petronas', 'has', 'performed', 'made', 'huge', 'profit'], ['i', 'feel', 'romantic']]
